# Experimento: CNN Baseline vs Baseline + Augmentation

Comparacao de custo computacional (tempo de treino) e metricas (acuracia, loss)
entre o modelo baseline sem augmentation e com augmentation ativado.

Augmentation layers: RandomFlip, RandomRotation, RandomContrast

## Como usar no Colab
1. Crie um Personal Access Token no GitLab (Settings > Access Tokens > scope: read_repository)
2. Execute a celula de setup abaixo e cole o token quando solicitado
3. Execute as celulas restantes em ordem

In [ ]:
# === SETUP: Clonar repo privado e instalar dependencias ===
import os
from getpass import getpass

GITLAB_HOST = "git.inteli.edu.br"
REPO_PATH = "graduacao/2026-1a/t11/g01.git"
BRANCH = "138-rodar-o-experimento-com-augmentation-ativado-coletar-tempos-de-execucao-para-comparar-custo"

if not os.path.exists("g01"):
    token = getpass("Cole seu GitLab Personal Access Token: ")
    !git clone --branch {BRANCH} https://oauth2:{token}@{GITLAB_HOST}/{REPO_PATH}
    del token  # nao manter token em memoria

os.chdir("g01")
!pip install -q pyyaml

In [ ]:
# === UPLOAD: pixels_dataset.csv (necessario pois .gitignore ignora *.csv) ===
from google.colab import files
import shutil

print("Faca upload do arquivo: data/pixels_dataset.csv")
uploaded = files.upload()

# Mover para o local esperado pela config
os.makedirs("data", exist_ok=True)
for filename in uploaded:
    shutil.move(filename, "data/pixels_dataset.csv")
    print(f"✓ {filename} movido para data/pixels_dataset.csv")

In [ ]:
import sys
sys.path.insert(0, ".")

from src.models.experiment_runner import run_multiple_experiments
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
configs = ["baseline", "baseline_augmented"]
results = run_multiple_experiments(configs)
results

In [ ]:
df = pd.DataFrame(results)
cols = [
    "config_name",
    "augmentation_enabled",
    "training_time_seconds",
    "final_train_acc",
    "final_val_acc",
    "final_train_loss",
    "final_val_loss",
    "epochs_run",
]
df_compare = df[cols].copy()
df_compare["training_time_seconds"] = df_compare["training_time_seconds"].round(2)
df_compare

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Tempo de treino
axes[0].bar(df_compare["config_name"], df_compare["training_time_seconds"], color=["steelblue", "coral"])
axes[0].set_title("Tempo de Treino (s)")
axes[0].set_ylabel("Segundos")

# Acuracia de validacao
axes[1].bar(df_compare["config_name"], df_compare["final_val_acc"], color=["steelblue", "coral"])
axes[1].set_title("Acuracia de Validacao")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)

# Loss de validacao
axes[2].bar(df_compare["config_name"], df_compare["final_val_loss"], color=["steelblue", "coral"])
axes[2].set_title("Loss de Validacao")
axes[2].set_ylabel("Loss")

plt.tight_layout()
plt.savefig("outputs/augmentation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Diferenca percentual no tempo
time_baseline = df_compare.loc[df_compare["config_name"] == "baseline", "training_time_seconds"].values[0]
time_augmented = df_compare.loc[df_compare["config_name"] == "baseline_augmented", "training_time_seconds"].values[0]
pct_diff = ((time_augmented - time_baseline) / time_baseline) * 100
print(f"\nDiferenca de tempo: {pct_diff:+.1f}% ({'mais lento' if pct_diff > 0 else 'mais rapido'} com augmentation)")